In [5]:
from typing import Optional
from pydantic import BaseModel
import arxiv


class Article(BaseModel):
    title: str
    summary: str
    published_date: str
    pdf_link: Optional[str]

    @classmethod
    def from_arxiv_result(cls, result: arxiv.Result) -> "Article":
        pdf_links = [str(link) for link in result.links if "/pdf/" in str(link)]
        pdf_link = pdf_links[0] if pdf_links else None

        return cls(
            title=result.title,
            summary=result.summary,
            published_date=result.published.strftime("%Y-%m-%d"),
            pdf_link=pdf_link,
        )

    def __str__(self):
        summary_snippet = "\n".join(self.summary.splitlines()[:3]) + "\n[...]"
        return f"title: {self.title}\ndate:{self.published_date}\npdf url:{self.pdf_link}\n\n{summary_snippet}"


def get_articles_content(query: str, max_results:int=3)->list[Article]:
    client = arxiv.Client()
    search = arxiv.Search(
        query=query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.Relevance
    )

    articles_iterator = map(Article.from_arxiv_result, client.results(search))
    articles_with_link = filter(lambda article:article.pdf_link is not None, articles_iterator)
    return list(articles_with_link)

def get_articles(query:str, max_results:int) -> list[Article]:
    print(f"search for {query} on arxiv")
    return get_articles_content(query, max_results)

In [17]:
# split
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_abstract(abstract, chunk_size=256, chunk_overlap=64):
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    return splitter.split_text(abstract)


In [ ]:
# reference
import re


def extract_references(text):
    pattern = r"\[(\d+)\]\s+[^\n]+"
    matches = re.findall(pattern, text)
    references = []
    for match in matches:
        ref_num, ref_text = match
        references.append({"ref_num": ref_num, "ref_text": ref_text})
    return references

In [16]:
# sample
abstract = """
Artificial intelligence (AI) predictions are increasingly used to inform human decisions1,2.
Here, using a behavioral implementation of the classic Newcomb’s paradox3 in 1,305 participants, 
we show that AI predictions can also shape the reasoning people use to make a decision. 
In this paradigm, perceived predictive authority can alter how people reason about their future actions, 
leading them to forgo a guaranteed reward. 
Over 40% of participants treated AI as such a predictive authority about their own behavior, 
significantly increasing the odds of forgoing the guaranteed reward by a factor of 3.39 (95% CI:2.45–4.70) 
and reducing earnings by 10.7–42.9%. 
The effect appeared across AI presentations and decision contexts and remained detectable even when predictions repeatedly failed. 
When people perceive AI as capable of predicting their personal behavior, 
the mere presence of AI predictions may shape their decision-making, narrowing the futures they
consider4

"""
chunks = split_abstract(abstract)
for idx, chunk in enumerate(chunks):
    print(f"chunk:{idx}\n{chunk}")

chunk:0
Artificial intelligence (AI) predictions are increasingly used to inform human decisions1,2.
Here, using a behavioral implementation of the classic Newcomb’s paradox3 in 1,305 participants,
chunk:1
we show that AI predictions can also shape the reasoning people use to make a decision. 
In this paradigm, perceived predictive authority can alter how people reason about their future actions, 
leading them to forgo a guaranteed reward.
chunk:2
leading them to forgo a guaranteed reward. 
Over 40% of participants treated AI as such a predictive authority about their own behavior, 
significantly increasing the odds of forgoing the guaranteed reward by a factor of 3.39 (95% CI:2.45–4.70)
chunk:3
and reducing earnings by 10.7–42.9%. 
The effect appeared across AI presentations and decision contexts and remained detectable even when predictions repeatedly failed. 
When people perceive AI as capable of predicting their personal behavior,
chunk:4
the mere presence of AI predictions may sha

In [ ]:
# test

articles = get_articles("AI Agent", max_results=3)
for article in articles:
    print(article)


search for AI Agent on arxiv
title: Faith in AI can narrow the futures individuals consider
date:2026-03-30
pdf url:https://arxiv.org/pdf/2603.28944v2\Artificial intelligence (AI) predictions are increasingly used to inform human decisions. Here, using a behavioral implementation of the classic Newcomb's paradox in 1,305 participants, we show that AI predictions can also shape the reasoning people use to make a decision. In this paradigm, perceived predictive authority can alter how people reason about their future actions, leading them to forgo a guaranteed reward. Over 40% of participants treated AI as such a predictive authority about their own behavior, significantly increasing the odds of forgoing the guaranteed reward by a factor of 3.39 (95% CI: 2.45-4.70) and reducing earnings by 10.7-42.9%. The effect appeared across AI presentations and decision contexts and remained detectable even when predictions repeatedly failed. When people perceive AI as capable of predicting their per

In [ ]:
# embedding

import pickle
import numpy as np
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="qwen3-embedding:0.6b", dimensions=512)

# vectors = embeddings.embed_documents(chunks)

# print(f"嵌入的文本块数量: {len(vectors)}")
# print(f"每个向量的维度: {len(vectors[0])}")
# print(f"第一个向量的前3个坐标: {vectors[0][:3]}")



嵌入的文本块数量: 5
每个向量的维度: 512
第一个向量的前3个坐标: [-0.099598154, 0.03695321, -0.004781956]


In [28]:
# save to es
from langchain_elasticsearch import ElasticsearchStore

metadatas = [
    {"paper_id": "2603.28944v2", "chunk_id": i} for i, chunk in enumerate(chunks)
]

# 会自动调用 embeddings.embed_documents() 并在 ES 中创建索引
vector_store = ElasticsearchStore.from_texts(
    texts=chunks,
    embedding=embeddings,
    metadatas=metadatas,
    index_name="demo_arxiv",
    es_url="http://192.168.1.7:9200",
    distance_strategy="COSINE",
)

In [36]:
# query

docs = vector_store.similarity_search_with_score(
    query="how many participants treated AI as such a predictive authority about their own behavior ?",
    k=3
)
for doc, score in docs:
    print(f"score:{score}\n{doc}\n\n")

score:0.88455117
page_content='leading them to forgo a guaranteed reward. 
Over 40% of participants treated AI as such a predictive authority about their own behavior, 
significantly increasing the odds of forgoing the guaranteed reward by a factor of 3.39 (95% CI:2.45–4.70)' metadata={'paper_id': '2603.28944v2', 'chunk_id': 2}


score:0.86512595
page_content='and reducing earnings by 10.7–42.9%. 
The effect appeared across AI presentations and decision contexts and remained detectable even when predictions repeatedly failed. 
When people perceive AI as capable of predicting their personal behavior,' metadata={'paper_id': '2603.28944v2', 'chunk_id': 3}


score:0.8542217
page_content='we show that AI predictions can also shape the reasoning people use to make a decision. 
In this paradigm, perceived predictive authority can alter how people reason about their future actions, 
leading them to forgo a guaranteed reward.' metadata={'paper_id': '2603.28944v2', 'chunk_id': 1}




In [39]:
# intention
from langchain_core.messages import SystemMessage, HumanMessage

import os
from datetime import datetime
from dotenv import load_dotenv
from uuid import uuid4

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
    model="deepseek-flash",
)


def classify_intent(query):
    messages = [
        SystemMessage(
            content="请判断用户学术查询属于哪一类意图（综述，方法对比，引用追溯，趋势分析，其他）；只返回类别名称"
        ),
        HumanMessage(content=query),
    ]
    response = llm.invoke(messages)
    return response.content.strip()


def rewrite_query(query, intent):
    system_prompt_template = f"""
            当前时间{datetime.now().strftime('%y-%m-%d %H:%M:%S')}
            针对学术检索，帮用户将以下查询更改为更具体、可检索的表达：
            查询内容:{query}
            查询意图：{intent}

            只返回改写后的query
            """
    messages = [
        SystemMessage(
            content=system_prompt_template.format(query=query, intent=intent)
        ),
        HumanMessage(content=query),
    ]
    response = llm.invoke(messages)
    return response.content.strip()

query = "最近半年AI Agent综述"
intent = classify_intent(query)
new_query = rewrite_query(query, intent)

print(f"query:{query}\nintent:{intent}\nnew query:{new_query}")

query:最近半年AI Agent综述
intent:综述
new query:("AI agent" OR "LLM agent" OR "large language model agent") AND (survey OR review) AND (2026-04-11 TO 2026-10-11)
